In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install -q -U "torchao>=0.16.0" peft accelerate

In [ ]:
import pandas as pd
import numpy as np
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    Trainer,
    TrainingArguments
)

from peft import LoraConfig, get_peft_model, TaskType

In [ ]:
train = pd.read_csv(
    "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)

print(train.shape)
train.head()

In [ ]:
label_map = {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "E": 4
}

train["label"] = train["answer"].map(label_map)

print("Label:", train.iloc[150]["label"])

In [ ]:
row = train.iloc[0]

option_b_input = (
    str(row["prompt"])
    + " [SEP] "
    + str(row["B"])
)

print(option_b_input)
print("Length:", len(option_b_input))

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)

In [ ]:
row = train.iloc[0]

choices = ["A", "B", "C", "D", "E"]

formatted_inputs = [
    str(row["prompt"]) + " [SEP] " + str(row[c])
    for c in choices
]

tokens = tokenizer(
    formatted_inputs,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

input_ids = tokens["input_ids"].unsqueeze(0)

print("Shape:", input_ids.shape)
print("Second dimension:", input_ids.shape[1])

In [ ]:
batch_inputs = []

for i in range(16):

    row = train.iloc[i]

    row_choices = [
        str(row["prompt"]) + " [SEP] " + str(row[c])
        for c in choices
    ]

    batch_inputs.extend(row_choices)


batch_tokens = tokenizer(
    batch_inputs,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

batch_input_ids = batch_tokens["input_ids"].reshape(
    16, 5, 128
)

print("Shape:", batch_input_ids.shape)
print("Total positions:", batch_input_ids.numel())

In [ ]:
mc_model = AutoModelForMultipleChoice.from_pretrained(
    "bert-base-uncased"
)

row = train.iloc[0]

formatted_inputs = [
    str(row["prompt"]) + " [SEP] " + str(row[c])
    for c in choices
]

tokens = tokenizer(
    formatted_inputs,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

mc_inputs = {
    "input_ids": tokens["input_ids"].unsqueeze(0),
    "attention_mask": tokens["attention_mask"].unsqueeze(0)
}

with torch.no_grad():
    output = mc_model(**mc_inputs)

print("Logits shape:", output.logits.shape)
print("Number of logits:", output.logits.shape[1])

In [ ]:
correct_label = torch.tensor([
    int(train.iloc[0]["label"])
])

output = mc_model(
    **mc_inputs,
    labels=correct_label
)

print("Loss:", output.loss)
print("Dimensions:", output.loss.dim())

In [ ]:
lora_model = AutoModelForMultipleChoice.from_pretrained(
    "bert-base-uncased"
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

lora_model = get_peft_model(
    lora_model,
    lora_config
)

trainable_params = sum(
    p.numel()
    for p in lora_model.parameters()
    if p.requires_grad
)

print("Trainable parameters:", trainable_params)

In [ ]:
data = []

for i in range(100):

    row = train.iloc[i]

    formatted_inputs = [
        str(row["prompt"]) + " [SEP] " + str(row[c])
        for c in choices
    ]

    encoded = tokenizer(
        formatted_inputs,
        padding="max_length",
        truncation=True,
        max_length=128
    )

    data.append({
        "input_ids": encoded["input_ids"],
        "attention_mask": encoded["attention_mask"],
        "labels": int(row["label"])
    })


hf_dataset = Dataset.from_list(data)

hf_dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels"
    ]
)

first_item = hf_dataset[0]

print("Shape:", first_item["input_ids"].shape)
print("Choices:", first_item["input_ids"].shape[0])

In [ ]:
tiny_data = []

for i in range(32):

    row = train.iloc[i]

    formatted_inputs = [
        str(row["prompt"]) + " [SEP] " + str(row[c])
        for c in choices
    ]

    encoded = tokenizer(
        formatted_inputs,
        padding="max_length",
        truncation=True,
        max_length=64
    )

    tiny_data.append({
        "input_ids": encoded["input_ids"],
        "attention_mask": encoded["attention_mask"],
        "labels": int(row["label"])
    })


tiny_dataset = Dataset.from_list(tiny_data)

tiny_dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels"
    ]
)

print(len(tiny_dataset))

In [ ]:
from transformers import set_seed

set_seed(42)

ft_model = AutoModelForMultipleChoice.from_pretrained(
    "bert-base-uncased"
)

ft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

ft_model = get_peft_model(
    ft_model,
    ft_config
)

In [ ]:
training_args = TrainingArguments(
    output_dir="./milestone4_output",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    learning_rate=5e-5,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
    seed=42
)

trainer = Trainer(
    model=ft_model,
    args=training_args,
    train_dataset=tiny_dataset
)

train_result = trainer.train()

print("Global step:", trainer.state.global_step)

In [ ]:
training_args = TrainingArguments(
    output_dir="./milestone4_output",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    learning_rate=5e-5,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
    seed=42
)

trainer = Trainer(
    model=ft_model,
    args=training_args,
    train_dataset=tiny_dataset
)

train_result = trainer.train()

print("Global step:", trainer.state.global_step)

In [ ]:
ft_model.eval()

row = train.iloc[0]

formatted_inputs = [
    str(row["prompt"]) + " [SEP] " + str(row[c])
    for c in choices
]

encoded = tokenizer(
    formatted_inputs,
    padding="max_length",
    truncation=True,
    max_length=64,
    return_tensors="pt"
)

device = next(ft_model.parameters()).device

input_ids = encoded["input_ids"].unsqueeze(0).to(device)
attention_mask = encoded["attention_mask"].unsqueeze(0).to(device)

with torch.no_grad():

    output = ft_model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

probabilities = torch.softmax(
    output.logits,
    dim=-1
)

print("Probabilities:", probabilities)

option_e_probability = probabilities[0, 4].item()

print(
    "Option E probability:",
    round(option_e_probability, 4)
)